## 1) Rutas del proyecto y del dataset (auto-detección)

Detectamos automáticamente la raíz del proyecto para poder ejecutar el notebook desde distintas carpetas.  
Definimos las rutas a `data/audio/raw` y la salida del manifest `data/audio/manifest.csv`.

In [1]:
from pathlib import Path
import pandas as pd
import re

# --- Detecta la raíz del proyecto aunque ejecutes desde /notebooks ---
here = Path.cwd().resolve()
PROJECT_ROOT = here
while not (PROJECT_ROOT / "data" / "audio" / "raw").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("No encuentro data/audio/raw. Ejecuta el notebook dentro del repo del proyecto.")
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA = PROJECT_ROOT / "data" / "audio"
RAW = DATA / "raw"
MANIFEST_PATH = DATA / "manifest.csv"

ravdess_root = RAW / "ravdess"
crema_root   = RAW / "crema_d" / "AudioWAV"
meld_root    = RAW / "meld"

print("CWD:", here)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW:", RAW)
print("ravdess_root exists?", ravdess_root.exists())
print("crema_root exists?", crema_root.exists())
print("meld_root exists?", meld_root.exists())

CWD: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\notebooks
PROJECT_ROOT: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion
RAW: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\raw
ravdess_root exists? True
crema_root exists? True
meld_root exists? True


## 2) RAVDESS → construir manifest (path, label, speaker)

Recorremos el dataset **RAVDESS** y extraemos la emoción desde el nombre del archivo (código numérico).  
Mapeamos cada código a una etiqueta estándar (`neutral`, `joy`, `sadness`, etc.) y guardamos:

- `path`: ruta del audio  
- `label`: emoción normalizada  
- `dataset`: nombre del dataset  
- `speaker`: actor/hablante (si se detecta)  
- `split`: se deja en `all` para este dataset

In [2]:
ravdess_emomap = {
    "01": "neutral",
    "02": "neutral",  # calm -> neutral
    "03": "joy",
    "04": "sadness",
    "05": "anger",
    "06": "fear",
    "07": "disgust",
    "08": "surprise",
}

rav_rows = []

for wav in ravdess_root.rglob("*.wav"):
    parts = wav.stem.split("-")
    if len(parts) >= 3:
        emo = parts[2]
        label = ravdess_emomap.get(emo)
        actor = None
        for p in wav.parts:
            if p.lower().startswith("actor_"):
                actor = p.split("_")[-1]
                break
        if label:
            rav_rows.append({
                "path": str(wav),
                "label": label,
                "dataset": "ravdess",
                "speaker": f"actor_{actor}" if actor else None,
                "split": "all"
            })

rav_df = pd.DataFrame(rav_rows)
print("RAVDESS:", len(rav_df))
rav_df.head()


RAVDESS: 1440


,path,label,dataset,speaker,split
0,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,neutral,ravdess,actor_01,all
1,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,neutral,ravdess,actor_01,all
2,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,neutral,ravdess,actor_01,all
3,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,neutral,ravdess,actor_01,all
4,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,neutral,ravdess,actor_01,all


## 3) CREMA-D → construir manifest (regex + etiquetas)

Recorremos **CREMA-D** y usamos una expresión regular para extraer:
- ID del speaker
- código de emoción (ANG, SAD, HAP, …)

Luego convertimos ese código a una etiqueta estándar y generamos el DataFrame `crema_df`.

In [3]:
crema_emomap = {
    "ANG": "anger",
    "DIS": "disgust",
    "FEA": "fear",
    "HAP": "joy",
    "NEU": "neutral",
    "SAD": "sadness",
}

crema_rows = []
pattern = re.compile(r"^(?P<spk>\d+)_.+_(?P<emo>[A-Z]{3})_.+\.wav$", re.IGNORECASE)

for wav in crema_root.rglob("*.wav"):
    m = pattern.match(wav.name)
    if not m:
        continue
    emo = m.group("emo").upper()
    label = crema_emomap.get(emo)
    if label:
        crema_rows.append({
            "path": str(wav),
            "label": label,
            "dataset": "crema_d",
            "speaker": f"crema_{m.group('spk')}",
            "split": "all"
        })

crema_df = pd.DataFrame(crema_rows)
print("CREMA-D:", len(crema_df))
crema_df.head()


CREMA-D: 7442


,path,label,dataset,speaker,split
0,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,anger,crema_d,crema_1001,all
1,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,disgust,crema_d,crema_1001,all
2,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,fear,crema_d,crema_1001,all
3,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,joy,crema_d,crema_1001,all
4,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,neutral,crema_d,crema_1001,all


## 4) MELD → unir etiquetas (CSV) con archivos multimedia

Para **MELD**:
1) Leemos los CSV de etiquetas (train/dev/test) y normalizamos la columna `Emotion`.  
2) Creamos una clave (`key`) tipo `dia<Dialogue_ID>_utt<Utterance_ID>` para emparejar con el nombre del archivo.  
3) Recorremos la carpeta de MELD buscando archivos de audio/vídeo y los unimos con su etiqueta y split.

In [4]:
import pandas as pd

def load_meld_labels(csv_path, split_name):
    df = pd.read_csv(csv_path)
    df["key"] = "dia" + df["Dialogue_ID"].astype(str) + "_utt" + df["Utterance_ID"].astype(str)
    df["split"] = split_name

    norm = {
        "joy": "joy",
        "sadness": "sadness",
        "anger": "anger",
        "fear": "fear",
        "disgust": "disgust",
        "surprise": "surprise",
        "neutral": "neutral",
    }
    df["label"] = df["Emotion"].astype(str).str.lower().map(norm)
    df = df.dropna(subset=["label"])
    return df[["key", "label", "split"]]

# ✅ rutas correctas (con .csv)
train_csv = meld_root / "train" / "train_sent_emo.csv"
dev_csv   = meld_root / "dev_sent_emo.csv"
test_csv  = meld_root / "test_sent_emo.csv"

print("MELD train labels:", train_csv.exists(), train_csv)
print("MELD dev labels:", dev_csv.exists(), dev_csv)
print("MELD test labels:", test_csv.exists(), test_csv)

meld_labels = []
if train_csv.exists(): meld_labels.append(load_meld_labels(train_csv, "train"))
if dev_csv.exists():   meld_labels.append(load_meld_labels(dev_csv, "dev"))
if test_csv.exists():  meld_labels.append(load_meld_labels(test_csv, "test"))

if not meld_labels:
    raise RuntimeError("No se encontraron CSVs de etiquetas de MELD. Revisa rutas.")

meld_lab_df = pd.concat(meld_labels, ignore_index=True)

media_ext = {".wav",".mp4",".m4a",".mp3",".flac",".ogg",".aac"}

meld_rows = []
for f in meld_root.rglob("*"):
    if f.is_file() and f.suffix.lower() in media_ext:
        key = f.stem
        hit = meld_lab_df[meld_lab_df["key"] == key]
        if len(hit) == 0:
            continue
        meld_rows.append({
            "path": str(f),
            "label": hit.iloc[0]["label"],
            "dataset": "meld",
            "speaker": None,
            "split": hit.iloc[0]["split"]
        })

meld_df = pd.DataFrame(meld_rows)
print("MELD samples encontradas:", len(meld_df))
meld_df.head()

MELD train labels: True C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\raw\meld\train\train_sent_emo.csv
MELD dev labels: True C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\raw\meld\dev_sent_emo.csv
MELD test labels: True C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\raw\meld\test_sent_emo.csv
MELD samples encontradas: 13716


,path,label,dataset,speaker,split
0,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,neutral,meld,None,train
1,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,neutral,meld,None,train
2,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,neutral,meld,None,train
3,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,neutral,meld,None,train
4,C:\Users\Rafa\Downloads\proyecto-ia-20260212T1...,neutral,meld,None,train


## 5) Unificar datasets y exportar `manifest.csv`

Unimos los tres DataFrames (`rav_df`, `crema_df`, `meld_df`) en un único manifest.  
Mostramos la distribución por dataset y emoción para comprobar el balanceo y finalmente guardamos:

**Salida:** `data/audio/manifest.csv`

In [5]:
df = pd.concat([rav_df, crema_df, meld_df], ignore_index=True)

print("TOTAL MUESTRAS:", len(df))
print("\nDistribución por dataset y emoción:\n")
print(df.groupby(["dataset","label"]).size().sort_values(ascending=False))

MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(MANIFEST_PATH, index=False)

print("\nGuardado en:", MANIFEST_PATH)


TOTAL MUESTRAS: 22598

Distribución por dataset y emoción:

dataset  label   
meld     neutral     6432
         joy         2392
         surprise    1679
         anger       1547
crema_d  joy         1271
         fear        1271
         disgust     1271
         anger       1271
         sadness     1271
         neutral     1087
meld     sadness      913
         fear         383
         disgust      370
ravdess  neutral      288
         disgust      192
         anger        192
         fear         192
         joy          192
         sadness      192
         surprise     192
dtype: int64

Guardado en: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\data\audio\manifest.csv
